<a href="https://colab.research.google.com/github/batu-uchicago/bayes-applications/blob/main/Bayesian%20Inference/credit_loan.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# The loan officer

An applicant asks for a loan.
Whether they are risky is hidden from the officer, who only sees how many late payments they have made: 0, 1 or 2.

| Applicant risky? | $P(h)$ | $P(\text{late} = 0 \mid h)$ | $P(\text{late} = 1 \mid h)$ | $P(\text{late} = 2 \mid h)$ |
|---|---|---|---|---|
| no | 0.6 | 0.7 | 0.2 | 0.1 |
| yes | 0.4 | 0.4 | 0.3 | 0.3 |

The officer can accept or reject, and each choice pays off differently depending on the hidden truth:

| $U(a, h)$ | not risky | risky |
|---|---|---|
| accept | 3 | −3 |
| reject | −2 | 0 |

The officer should take the action with the highest expected utility under the posterior:

$$a^* = \arg\max_a \sum_h U(a, h)\, P(h \mid \text{late})$$

## 1. Setup

Run the cell below once. On Colab it installs pgmpy.

In [1]:
# On your laptop, `pip install -r requirements.txt` (see the README) has installed everything.
import subprocess, sys

if "google.colab" in sys.modules:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pgmpy>=1.1.2,<1.3"], check=True)
print("Setup complete.")

Setup complete.


In [2]:
import warnings

# requirements.txt keeps pgmpy below 1.3, so hide its notices about classes that move in 1.3.
warnings.filterwarnings("ignore", message=r".*will be removed in v1\.3\.0", category=FutureWarning)

from pgmpy.factors.discrete import TabularCPD
from pgmpy.inference import VariableElimination
from pgmpy.models import DiscreteBayesianNetwork

## 2. The model: a hidden risk and the late payments it causes

In [3]:
model = DiscreteBayesianNetwork([("risky", "late_payments")])
model.add_cpds(
    TabularCPD("risky", 2, [[0.6], [0.4]], state_names={"risky": ["no", "yes"]}),
    TabularCPD(
        "late_payments", 3,
        [[0.7, 0.4],   # 0 late payments, if not risky | if risky
         [0.2, 0.3],   # 1
         [0.1, 0.3]],  # 2
        evidence=["risky"], evidence_card=[2],
        state_names={"late_payments": [0, 1, 2], "risky": ["no", "yes"]},
    ),
)
assert model.check_model()
print(model.get_cpds("late_payments"))

+------------------+-----------+------------+
| risky            | risky(no) | risky(yes) |
+------------------+-----------+------------+
| late_payments(0) | 0.7       | 0.4        |
+------------------+-----------+------------+
| late_payments(1) | 0.2       | 0.3        |
+------------------+-----------+------------+
| late_payments(2) | 0.1       | 0.3        |
+------------------+-----------+------------+


## 3. The belief: is this applicant risky?

The applicant has made two late payments. Bayes' rule turns the prior 0.4 into a posterior:

$$P(\text{risky} \mid \text{late} = 2) = \frac{0.3 \times 0.4}{0.1 \times 0.6 + 0.3 \times 0.4}$$

In [4]:
infer = VariableElimination(model)
belief = infer.query(["risky"], evidence={"late_payments": 2}, show_progress=False)
print(belief)

+------------+--------------+
| risky      |   phi(risky) |
+============+==============+
| risky(no)  |       0.3333 |
+------------+--------------+
| risky(yes) |       0.6667 |
+------------+--------------+


## 4. The decision: maximize the expected utility

In [5]:
utility = {
    "accept": {"no": 3, "yes": -3},
    "reject": {"no": -2, "yes": 0},
}


def expected_utility(action, belief):
    """Average the payoff of an action over the posterior on the hidden risk."""
    return sum(utility[action][h] * belief.get_value(risky=h) for h in ["no", "yes"])


for action in utility:
    print(f"EU({action}) = {expected_utility(action, belief):+.2f}")
print("The officer should", max(utility, key=lambda a: expected_utility(a, belief)))

EU(accept) = -1.00
EU(reject) = -0.67
The officer should reject


## 5. Every applicant at once

The same rule, for each number of late payments the officer might see.

In [6]:
print("late payments  P(risky | late)  EU(accept)  EU(reject)  decision")
for late in [0, 1, 2]:
    b = infer.query(["risky"], evidence={"late_payments": late}, show_progress=False)
    eu = {a: expected_utility(a, b) for a in utility}
    print(f"{late:13d}  {b.get_value(risky='yes'):15.2f}  {eu['accept']:+10.2f}  "
          f"{eu['reject']:+10.2f}  {max(eu, key=eu.get)}")

late payments  P(risky | late)  EU(accept)  EU(reject)  decision
            0             0.28       +1.34       -1.45  accept
            1             0.50       +0.00       -1.00  accept
            2             0.67       -1.00       -0.67  reject


## Try it

Change the prior `[[0.6], [0.4]]` in section 2 and rerun the notebook.
How high must $P(\text{risky})$ be before a single late payment is enough to reject?
Then change the utilities instead: what does a bank that hates losing good customers look like?